In [1]:
import pyvista as pv
pv.set_jupyter_backend('client')

import numpy as np
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack

from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN

from astropy.io import fits
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.style.use('dark_background')
import seaborn as sns

In [2]:
void_id = 2101900007889

center = np.array([-1394.06830, 1797.40756, 556.78357])
Reff = 92.49894

In [3]:
members = Table.read('/pscratch/sd/v/vtorresg/cosmic-web/voids_3d/extracted_members/LRG_NGC_void_2101900007889/random_members.fits')

members = members[(members['VOID_ID'] == void_id) & (members['MEMBER'] == True)]

member_points = np.column_stack([members['XCART'], members['YCART'], members['ZCART']]).astype(float)
len(member_points)

124

In [4]:
member_norm = (member_points - center) / Reff

cloud_members = pv.PolyData(member_points)

In [12]:
pv.set_jupyter_backend('client')

p = pv.Plotter(notebook=True, window_size=(1000, 800))

p.add_points(cloud_members, point_size=10, render_points_as_spheres=True, color='deepskyblue')
# p.add_points(np.array([[0., 0., 0.]]), point_size=15, render_points_as_spheres=True, color='red')

p.add_axes()
p.show_grid()

p.show()

Widget(value='<iframe src="/user/vtorresg/perlmutter-exclusive-node-cpu/proxy/38375/index.html?ui=P_0x7fb8f3b5…

In [13]:
tree = cKDTree(member_norm)

dist, _ = tree.query(member_norm, k=2)
dnn = dist[:, 1]

print('NN median =', np.median(dnn))
print('NN percentiles =', np.percentile(dnn, [10, 25, 50, 75, 90]))

NN median = 0.23830054436470935
NN percentiles = [0.12853505 0.18537104 0.23830054 0.3236449  0.42715674]


In [14]:
alpha = 2.0 * np.median(dnn)

volume = cloud_members.delaunay_3d(alpha=0.6)

surface = (volume.extract_surface(algorithm=None).clean().triangulate())

print('alpha =', alpha)
print(surface)

alpha = 0.4766010887294187
PolyData (0x7fb8ead8b7c0)
  N Cells:    0
  N Points:   124
  N Strips:   0
  X Bounds:   -2.250e+03, -1.907e+03
  Y Bounds:   2.523e+03, 2.843e+03
  Z Bounds:   6.597e+02, 9.644e+02
  N Arrays:   2


In [15]:
p = pv.Plotter(notebook=True, window_size=(1000, 800))

p.add_mesh(surface, color='deepskyblue', opacity=0.20,
           # show_edges=True
           )
p.add_points(cloud_members, color='white', point_size=2, render_points_as_spheres=True)

p.add_axes()
p.show_grid()

p.show()

Widget(value='<iframe src="/user/vtorresg/perlmutter-exclusive-node-cpu/proxy/38375/index.html?ui=P_0x7fb8e014…

In [16]:
xyz = np.column_stack([np.asarray(members['XCART'], dtype=float),
                       np.asarray(members['YCART'], dtype=float),
                       np.asarray(members['ZCART'], dtype=float)])

center = xyz.mean(axis=0)

xyz_norm = (xyz - center) / Reff

In [17]:
cloud = pv.PolyData(xyz_norm)

alpha = 0.6
volume = cloud.delaunay_3d(alpha=alpha)
surface = (volume.extract_surface(algorithm=None).clean().triangulate())

sphere = pv.Sphere(radius=1.0, center=(0, 0, 0), theta_resolution=100, phi_resolution=100)
center_point = pv.PolyData(np.array([[0.0, 0.0, 0.0]]))

/tmp/ipykernel_1834381/3602937852.py:5: InvalidMeshWarning: PolyData mesh is not valid:
 - Mesh has 1 cell array with incorrect length (length must be 300). Invalid array: 'vtkOriginalCellIds' (592)
  surface = (volume.extract_surface(algorithm=None).clean().triangulate())


In [19]:
p = pv.Plotter(notebook=True, window_size=(1000, 800))

p.add_mesh(surface, color='deepskyblue', opacity=0.30, show_edges=True, label='Void surface')

# random
p.add_points(cloud, point_size=4, render_points_as_spheres=True, color='cyan', label='Random members')

p.add_mesh(sphere, color='red', opacity=0.03)
p.add_mesh(sphere, color='red', style='wireframe', line_width=2, opacity=0.9, label=r'$R_{\rm eff}$ sphere')

# centro
p.add_points(center_point, point_size=18, render_points_as_spheres=True, color='yellow', label='Mean center')

p.add_axes()
p.show_grid()
p.add_legend()
p.reset_camera()

p.show()

2026-10-01 19:31:20.700 ( 130.787s) [    7FBA91F19B80]         vtkDataSet.cxx:746   WARN| vtkPolyData (0x562648588e40): Cell array vtkOriginalCellIds with 1 components, has 592 tuples but there are only 300 cells
2026-10-01 19:31:20.705 ( 130.793s) [    7FBA91F19B80]         vtkDataSet.cxx:746   WARN| vtkPolyData (0x562648588e40): Cell array vtkOriginalCellIds with 1 components, has 592 tuples but there are only 300 cells
2026-10-01 19:31:20.710 ( 130.798s) [    7FBA91F19B80]         vtkDataSet.cxx:746   WARN| vtkPolyData (0x562648588e40): Cell array vtkOriginalCellIds with 1 components, has 592 tuples but there are only 300 cells
2026-10-01 19:31:20.716 ( 130.803s) [    7FBA91F19B80]         vtkDataSet.cxx:746   WARN| vtkPolyData (0x562648588e40): Cell array vtkOriginalCellIds with 1 components, has 592 tuples but there are only 300 cells
2026-10-01 19:31:20.863 ( 130.950s) [    7FBA91F19B80]         vtkDataSet.cxx:746   WARN| vtkPolyData (0x562648588e40): Cell array vtkOriginalCellId

Widget(value='<iframe src="/user/vtorresg/perlmutter-exclusive-node-cpu/proxy/38375/index.html?ui=P_0x7fb8e014…

------

In [ ]:
ls = []

for it in range(100):
    t = Table.read(f'/pscratch/sd/v/vtorresg/cosmic-web/dr1/classification/lrg/ngc/zone_NGC_LRG_iter0{it:02d}.fits.gz')
    rand = t[t['ISDATA']==False]
    rand['r'] = (rand['NDATA'] - rand['NRAND']) / (rand['NDATA'] + rand['NRAND'])
    r_rand = rand[rand['r']<-0.25]
    ls.append(r_rand)
    # print(len(r_rand))

len(ls)

In [ ]:
tab = vstack(ls, join_type='outer')
len(tab)

In [4]:
raw = Table.read('/pscratch/sd/v/vtorresg/cosmic-web/dr1/raw/zone_NGC_LRG.fits.gz')
raw = raw[raw['RANDITER']!=-1]

# raw = raw[raw['RANDITER']<10]
len(raw)

147613500

In [5]:
np.unique(raw['RANDITER'])

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99],
      dtype='>i2')

In [ ]:
vtab = join(raw, tab, keys='TARGETID', join_type='inner')
len(vtab)

In [ ]:
voidcat = Table.read('/pscratch/sd/v/vtorresg/cosmic-web/dr1/groups/voids_LRG_NGC.fits')
voidcat

In [ ]:
void_id = 2101900007889

row = voidcat[voidcat['VOID_ID'] == void_id][0]

center = np.array([row['X'], row['Y'], row['Z']])

eigvals = np.array([row['EIGVAL_1'], row['EIGVAL_2'], row['EIGVAL_3']])

eigvecs = np.array([[row['EIGVEC_1_X'], row['EIGVEC_1_Y'], row['EIGVEC_1_Z']],
                    [row['EIGVEC_2_X'], row['EIGVEC_2_Y'], row['EIGVEC_2_Z']],
                    [row['EIGVEC_3_X'], row['EIGVEC_3_Y'], row['EIGVEC_3_Z']]])
V = row['V']

In [ ]:
order = np.argsort(eigvals)[::-1]

eigvals = eigvals[order]
eigvecs = eigvecs[order]

In [ ]:
q = np.sqrt(eigvals)

In [ ]:
s = (3 * V / (4 * np.pi * np.prod(q)))**(1/3)

axes = s * q
a, b, c = axes
a,b,c

In [ ]:
sphere = pv.Sphere(radius=1.0, theta_resolution=100, phi_resolution=100)

pts = sphere.points.copy()

pts[:, 0] *= a
pts[:, 1] *= b
pts[:, 2] *= c

pts = pts @ eigvecs

pts += center

ellipsoid = sphere.copy()
ellipsoid.points = pts

In [ ]:
p = pv.Plotter(notebook=True, off_screen=True, window_size=(900, 700))

p.add_mesh(ellipsoid, opacity=0.5, show_edges=False)

p.add_axes()
p.show_grid()

p.show(jupyter_backend='static')

In [ ]:
raw_all = Table.read('/pscratch/sd/v/vtorresg/cosmic-web/dr1/raw/zone_NGC_LRG.fits.gz')
raw_all = raw_all[raw_all['RANDITER']==-1]
raw_all.colnames

In [ ]:
!ls /pscratch/sd/v/vtorresg/cosmic-web/dr1/probabilities/lrg/ngc

In [ ]:
Table.read('/pscratch/sd/v/vtorresg/cosmic-web/dr1/probabilities/lrg/ngc/zone_NGC_LRG_probability_iterdata.fits.gz')